In [41]:
#%pip install pandas numpy faker
#%pip install faker faker-vehicle


In [42]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import random
import string
from faker_vehicle import VehicleProvider

In [43]:
fake = Faker('en_US')
random.seed(42)
Faker.seed(42)

num_records = 200000


In [44]:
fake.address()

'433 Jill Springs\nNew Roberttown, CO 29158'

In [45]:
def introduce_noise(data, error_rate=0.05, null_rate=0.0):
    noisy_data = []
    for val in data:
        rand = random.random()
        if rand < null_rate:
            noisy_data.append(None)
        elif rand < null_rate + error_rate:
            noisy_data.append("ERROR")
        else:
            noisy_data.append(val)
    return noisy_data

def introduce_date_noise(dates, error_rate=0.05):
    noisy_dates = []
    for date in dates:
        rand = random.random()
        if rand < error_rate / 3:
            noisy_dates.append(date.strftime('%d/%m/%y'))
        elif rand < 2 * error_rate / 3:
            noisy_dates.append(date.strftime('%m/%d/%y'))
        # elif rand < error_rate:
        #     noisy_dates.append(date.strftime('%y/%m/%d'))
        # else:
        #     noisy_dates.append(date.strftime('%Y-%m-%d'))
        else:
            noisy_dates.append(date.strftime('%b/%y'))
    return noisy_dates

def generate_random_part_numbers(prefix, count):
    part_numbers = set()
    while len(part_numbers) < count:
        suffix = ''.join(random.choices(string.digits, k=3))
        part_numbers.add(f"{prefix}{suffix}")
    return list(part_numbers)

In [46]:
# --- Part Catalog ---
part_catalog = {
    "Spark Plug": generate_random_part_numbers("SP6",22),
    "Fuel Pump": generate_random_part_numbers("FP7", 20),
    "Oil Filter": generate_random_part_numbers("OF8", 20),
    "Air Filter": generate_random_part_numbers("AF9", 20),
    "Ignition Coil": generate_random_part_numbers("IC0", 18),
    "Brake Rotor": generate_random_part_numbers("BR5", 12),
    "Wiper Blade": generate_random_part_numbers("WB3", 15),
    "Tailgate Lift Support": generate_random_part_numbers("TLS4", 10),
    "Engine Water Pump": generate_random_part_numbers("EWP1", 10),
}

In [47]:

# Generate parts and engine data
part_types = []
part_numbers = []
for _ in range(num_records):
    part = random.choice(list(part_catalog.keys()))
    part_types.append(part)
    part_numbers.append(random.choice(part_catalog[part]))

cc_cluster_1 = np.random.normal(loc=1400, scale=200, size=num_records // 2)
cc_cluster_2 = np.random.normal(loc=3500, scale=500, size=num_records // 2)
engine_cc = np.concatenate([cc_cluster_1, cc_cluster_2])
np.random.shuffle(engine_cc)
engine_cc = np.clip(engine_cc, 600, 6000)


# Base IDs
invoice_ids = [f"INV{str(i).zfill(6)}" for i in range(1, num_records + 1)]
customer_ids = [f"CUST{str(i % 10000).zfill(4)}" for i in range(1, num_records + 1)]

# Base distributions
service_costs = np.random.lognormal(mean=7.0, sigma=0.6, size=num_records)
odometers = np.random.lognormal(mean=10.0, sigma=0.5, size=num_records)
fuel_consumption = np.random.normal(loc=7.5, scale=1.5, size=num_records)
engine_temp = np.random.normal(loc=90, scale=10, size=num_records)

repair_duration = np.random.normal(loc=3, scale=1.5, size=num_records)
number_of_visits = np.random.poisson(lam=2, size=num_records)
parts_cost =  [round(random.uniform(5, 100), 2) for part in part_numbers]
labor_cost = np.random.normal(loc=10, scale=3, size=num_records)
discount_amount = np.random.normal(loc=50, scale=30, size=num_records)
insurance_coverage = np.random.uniform(low=70, high=100, size=num_records)
service_rating = np.random.normal(loc=4.2, scale=0.8, size=num_records)

# Clip ranges to make values realistic
fuel_consumption = np.clip(fuel_consumption, 2, 30)
engine_temp = np.clip(engine_temp, 50, 200)
repair_duration = np.clip(repair_duration, 0.5, 15)
# labor_cost = np.clip(labor_cost, 50, 1000)
discount_amount = np.clip(discount_amount, 0, 500)
service_rating = np.clip(service_rating, 1, 5)



# Initialize Faker
fake = Faker()
fake.add_provider(VehicleProvider)

# List of sample customers
customers = ["NAPA", "AutoZone", "O'Reilly", "Advance Auto Parts", "CarQuest", "Summit Racing", "RockAuto","Amazon", "Walmart"]


make = []
model = []
year = []
customer = []

for _ in range(num_records):
    vehicle = fake.vehicle_object()
    make.append(vehicle['Make'])
    model.append(vehicle['Model'])
    year.append(vehicle['Year'])
    customer.append(random.choice(customers))  # Random customer

In [48]:
vehicle = fake.vehicle_object()
print(vehicle)


{'Year': 2018, 'Make': 'Alfa Romeo', 'Model': 'Stelvio', 'Category': 'SUV'}


In [49]:
# Data Dictionary
data = {
    "Invoice_ID": invoice_ids,
    "Customer_ID": customer_ids,
    "Customer_Name": [fake.name() for _ in range(num_records)],
    # "Country": ["USA"] * num_records,
    "State": [fake.state() for _ in range(num_records)],

    "Vehicle_Year": year,
    "Vehilce_Make": make,
    "Vehicle_Model": model,
    "Customer": customer,
    "Purchase_Date": introduce_date_noise([fake.date_between(start_date='-5y', end_date='today') for _ in range(num_records)]),
    "Service_Date": introduce_date_noise([fake.date_between(start_date='-2y', end_date='today') for _ in range(num_records)]),
    "Warranty_Expiry": introduce_date_noise([fake.date_between(start_date='today', end_date='+5y') for _ in range(num_records)]),

    # "Service_Cost_USD": introduce_noise([round(val, 2) for val in service_costs]),
    "Odometer_km": introduce_noise([int(val) for val in odometers]),
    "Fuel_Consumption_L_per_100km": introduce_noise([round(val, 2) for val in fuel_consumption]),
    "Engine_CC": introduce_noise([int(round(cc / 100) * 100) for cc in engine_cc]),
    "Engine_Temp_C": introduce_noise([round(val, 1) for val in engine_temp]),
    "Feedback_Score": introduce_noise([random.choice([1, 2, 3, 4, 5]) for _ in range(num_records)]),
    "Payment_Method": [random.choice(["Credit Card", "Cash", "Online Transfer", "Cheque"]) for _ in range(num_records)],
    "Part_Type": part_types,
    "Part_Number": part_numbers,

    "Repair_Duration_Hours": introduce_noise([round(val, 2) for val in repair_duration]),
    # "Number_of_Visits": introduce_noise([int(val) for val in number_of_visits]),
    #"Parts_Cost_USD": introduce_noise([round(val, 2) for val in parts_cost]),
    "Labor_Cost_USD": introduce_noise([round(val, 2) for val in labor_cost]),
    "Discount_Amount_USD": introduce_noise([round(val, 2) for val in discount_amount]),
    "Insurance_Coverage_Percent": introduce_noise([round(val, 2) for val in insurance_coverage]),
    "Service_Rating": introduce_noise([round(val, 2) for val in service_rating]),
}

In [50]:
df = pd.DataFrame(data)
df

,Invoice_ID,Customer_ID,Customer_Name,State,Vehicle_Year,Vehilce_Make,Vehicle_Model,Customer,Purchase_Date,Service_Date,...,Engine_Temp_C,Feedback_Score,Payment_Method,Part_Type,Part_Number,Repair_Duration_Hours,Labor_Cost_USD,Discount_Amount_USD,Insurance_Coverage_Percent,Service_Rating
0,INV000001,CUST0001,Lance Hoffman,Texas,2003,Dodge,Ram 1500 Quad Cab,O'Reilly,Dec/21,Sep/23,...,82.7,ERROR,Online Transfer,Tailgate Lift Support,TLS4963,2.99,6.34,46.28,ERROR,3.89
1,INV000002,CUST0002,Meredith Barnes,Iowa,1993,Buick,Park Avenue,CarQuest,Oct/21,Dec/23,...,94.8,5,Cash,Engine Water Pump,EWP1486,ERROR,10.83,81.65,79.47,4.56
2,INV000003,CUST0003,Donald Booth,Maine,2010,Chevrolet,Silverado 1500 Regular Cab,Advance Auto Parts,Nov/21,Jan/25,...,91.9,4,Online Transfer,Engine Water Pump,EWP1480,3.19,7.35,40.05,91.96,5.0
3,INV000004,CUST0004,Caitlin Henderson,North Carolina,2008,Ford,Fusion,Walmart,Apr/23,Nov/24,...,92.2,ERROR,Online Transfer,Oil Filter,OF8528,5.76,12.19,29.63,98.23,5.0
4,INV000005,CUST0005,Daniel Gallagher,Kansas,2011,Scion,xB,RockAuto,Nov/24,May/24,...,79.2,2,Credit Card,Tailgate Lift Support,TLS4496,5.43,11.44,31.56,72.39,ERROR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,INV199996,CUST9996,Dustin Wade,Kansas,2006,Nissan,Xterra,O'Reilly,Apr/22,Mar/25,...,84.0,2,Online Transfer,Spark Plug,SP6220,2.45,10.09,65.06,96.73,4.87
199996,INV199997,CUST9997,Lindsey Sanchez,Oklahoma,2010,Chevrolet,Aveo,NAPA,Sep/23,Feb/25,...,99.5,1,Credit Card,Brake Rotor,BR5198,4.53,8.99,69.89,73.86,4.12
199997,INV199998,CUST9998,Kyle Edwards,Alaska,1999,MAZDA,626,Amazon,Jan/24,Apr/24,...,87.5,4,Cheque,Air Filter,AF9981,4.5,15.34,13.61,ERROR,4.41
199998,INV199999,CUST9999,Matthew Lamb,Idaho,2009,Ford,F250 Super Duty Crew Cab,AutoZone,May/21,Oct/23,...,88.8,5,Cash,Spark Plug,SP6570,1.99,7.4,101.65,90.89,4.73


In [51]:
unique_parts = df['Part_Number'].unique()
cost_mapping = {part: round(random.uniform(10, 100), 2) for part in unique_parts}
df['Parts_Cost_USD'] = df['Part_Number'].map(cost_mapping)

In [52]:
df

,Invoice_ID,Customer_ID,Customer_Name,State,Vehicle_Year,Vehilce_Make,Vehicle_Model,Customer,Purchase_Date,Service_Date,...,Feedback_Score,Payment_Method,Part_Type,Part_Number,Repair_Duration_Hours,Labor_Cost_USD,Discount_Amount_USD,Insurance_Coverage_Percent,Service_Rating,Parts_Cost_USD
0,INV000001,CUST0001,Lance Hoffman,Texas,2003,Dodge,Ram 1500 Quad Cab,O'Reilly,Dec/21,Sep/23,...,ERROR,Online Transfer,Tailgate Lift Support,TLS4963,2.99,6.34,46.28,ERROR,3.89,51.79
1,INV000002,CUST0002,Meredith Barnes,Iowa,1993,Buick,Park Avenue,CarQuest,Oct/21,Dec/23,...,5,Cash,Engine Water Pump,EWP1486,ERROR,10.83,81.65,79.47,4.56,19.74
2,INV000003,CUST0003,Donald Booth,Maine,2010,Chevrolet,Silverado 1500 Regular Cab,Advance Auto Parts,Nov/21,Jan/25,...,4,Online Transfer,Engine Water Pump,EWP1480,3.19,7.35,40.05,91.96,5.0,36.75
3,INV000004,CUST0004,Caitlin Henderson,North Carolina,2008,Ford,Fusion,Walmart,Apr/23,Nov/24,...,ERROR,Online Transfer,Oil Filter,OF8528,5.76,12.19,29.63,98.23,5.0,78.09
4,INV000005,CUST0005,Daniel Gallagher,Kansas,2011,Scion,xB,RockAuto,Nov/24,May/24,...,2,Credit Card,Tailgate Lift Support,TLS4496,5.43,11.44,31.56,72.39,ERROR,27.63
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,INV199996,CUST9996,Dustin Wade,Kansas,2006,Nissan,Xterra,O'Reilly,Apr/22,Mar/25,...,2,Online Transfer,Spark Plug,SP6220,2.45,10.09,65.06,96.73,4.87,63.13
199996,INV199997,CUST9997,Lindsey Sanchez,Oklahoma,2010,Chevrolet,Aveo,NAPA,Sep/23,Feb/25,...,1,Credit Card,Brake Rotor,BR5198,4.53,8.99,69.89,73.86,4.12,96.94
199997,INV199998,CUST9998,Kyle Edwards,Alaska,1999,MAZDA,626,Amazon,Jan/24,Apr/24,...,4,Cheque,Air Filter,AF9981,4.5,15.34,13.61,ERROR,4.41,94.23
199998,INV199999,CUST9999,Matthew Lamb,Idaho,2009,Ford,F250 Super Duty Crew Cab,AutoZone,May/21,Oct/23,...,5,Cash,Spark Plug,SP6570,1.99,7.4,101.65,90.89,4.73,56.51


In [53]:
df[['Invoice_ID', 'Customer_ID', 'Customer_Name', 'State', 'Customer','Vehicle_Year','Vehilce_Make', 'Vehicle_Model','Engine_CC', 'Purchase_Date','Warranty_Expiry', 'Odometer_km','Fuel_Consumption_L_per_100km',  'Engine_Temp_C', 'Service_Date','Part_Type', 'Part_Number','Repair_Duration_Hours', 'Parts_Cost_USD', 'Labor_Cost_USD', 'Discount_Amount_USD','Insurance_Coverage_Percent', 'Service_Rating','Feedback_Score', 'Payment_Method']]

,Invoice_ID,Customer_ID,Customer_Name,State,Customer,Vehicle_Year,Vehilce_Make,Vehicle_Model,Engine_CC,Purchase_Date,...,Part_Type,Part_Number,Repair_Duration_Hours,Parts_Cost_USD,Labor_Cost_USD,Discount_Amount_USD,Insurance_Coverage_Percent,Service_Rating,Feedback_Score,Payment_Method
0,INV000001,CUST0001,Lance Hoffman,Texas,O'Reilly,2003,Dodge,Ram 1500 Quad Cab,1300,Dec/21,...,Tailgate Lift Support,TLS4963,2.99,51.79,6.34,46.28,ERROR,3.89,ERROR,Online Transfer
1,INV000002,CUST0002,Meredith Barnes,Iowa,CarQuest,1993,Buick,Park Avenue,3500,Oct/21,...,Engine Water Pump,EWP1486,ERROR,19.74,10.83,81.65,79.47,4.56,5,Cash
2,INV000003,CUST0003,Donald Booth,Maine,Advance Auto Parts,2010,Chevrolet,Silverado 1500 Regular Cab,1200,Nov/21,...,Engine Water Pump,EWP1480,3.19,36.75,7.35,40.05,91.96,5.0,4,Online Transfer
3,INV000004,CUST0004,Caitlin Henderson,North Carolina,Walmart,2008,Ford,Fusion,1400,Apr/23,...,Oil Filter,OF8528,5.76,78.09,12.19,29.63,98.23,5.0,ERROR,Online Transfer
4,INV000005,CUST0005,Daniel Gallagher,Kansas,RockAuto,2011,Scion,xB,1200,Nov/24,...,Tailgate Lift Support,TLS4496,5.43,27.63,11.44,31.56,72.39,ERROR,2,Credit Card
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,INV199996,CUST9996,Dustin Wade,Kansas,O'Reilly,2006,Nissan,Xterra,1100,Apr/22,...,Spark Plug,SP6220,2.45,63.13,10.09,65.06,96.73,4.87,2,Online Transfer
199996,INV199997,CUST9997,Lindsey Sanchez,Oklahoma,NAPA,2010,Chevrolet,Aveo,3600,Sep/23,...,Brake Rotor,BR5198,4.53,96.94,8.99,69.89,73.86,4.12,1,Credit Card
199997,INV199998,CUST9998,Kyle Edwards,Alaska,Amazon,1999,MAZDA,626,1600,Jan/24,...,Air Filter,AF9981,4.5,94.23,15.34,13.61,ERROR,4.41,4,Cheque
199998,INV199999,CUST9999,Matthew Lamb,Idaho,AutoZone,2009,Ford,F250 Super Duty Crew Cab,3200,May/21,...,Spark Plug,SP6570,1.99,56.51,7.4,101.65,90.89,4.73,5,Cash


In [54]:
CustomerNames=df.Customer.unique()

In [55]:
for i in range (len(CustomerNames)):
    df[df['Customer']==CustomerNames[i]].to_csv(rf"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\WarrantyData\Warranty\{CustomerNames[i]}_WarrantyData.csv", index=False)
    print(f"✅ Dataset saved as '{CustomerNames[i]}_WarrantyData.csv'")


✅ Dataset saved as 'O'Reilly_WarrantyData.csv'
✅ Dataset saved as 'CarQuest_WarrantyData.csv'
✅ Dataset saved as 'Advance Auto Parts_WarrantyData.csv'
✅ Dataset saved as 'Walmart_WarrantyData.csv'
✅ Dataset saved as 'RockAuto_WarrantyData.csv'
✅ Dataset saved as 'AutoZone_WarrantyData.csv'
✅ Dataset saved as 'NAPA_WarrantyData.csv'
✅ Dataset saved as 'Amazon_WarrantyData.csv'
✅ Dataset saved as 'Summit Racing_WarrantyData.csv'


In [56]:
sales_records = 200000

In [57]:
customer_sales = []
part_types = []
part_numbers = []
for _ in range(sales_records):
    part = random.choice(list(part_catalog.keys()))
    part_types.append(part)
    part_numbers.append(random.choice(part_catalog[part]))
    customer_sales.append(random.choice(customers))

In [58]:
sales_data={
    "Part_Type": part_types,
    "Part_Number": part_numbers,
    "Customer": customer_sales,
    "Purchase_Date":[fake.date_between(start_date='-5y', end_date='today') for _ in range(sales_records)],
    "Units_Sold": np.random.poisson(lam=5, size=sales_records),
}

In [59]:
df_sales=pd.DataFrame(sales_data)
df_sales

,Part_Type,Part_Number,Customer,Purchase_Date,Units_Sold
0,Oil Filter,OF8723,RockAuto,2021-10-14,6
1,Engine Water Pump,EWP1486,O'Reilly,2022-10-11,5
2,Wiper Blade,WB3387,Summit Racing,2025-04-13,4
3,Oil Filter,OF8026,Amazon,2020-07-28,5
4,Ignition Coil,IC0980,NAPA,2024-08-07,5
...,...,...,...,...,...
199995,Oil Filter,OF8043,RockAuto,2022-11-29,5
199996,Tailgate Lift Support,TLS4497,Amazon,2023-04-04,5
199997,Tailgate Lift Support,TLS4638,Walmart,2021-07-22,5
199998,Engine Water Pump,EWP1028,Amazon,2021-09-04,7


In [60]:
df_sales['Purchase_Date'] = pd.to_datetime(df_sales['Purchase_Date'])
df_sales['Month'] = df_sales['Purchase_Date'].dt.strftime('%Y-%m') # Format as 'YYYY-MM' for consistent ordering

In [61]:
pivot_df = pd.pivot_table(df_sales, 
                            values='Units_Sold', 
                            index=["Part_Type",'Part_Number',"Customer"], 
                            columns='Month', 
                            aggfunc='sum', 
                            fill_value=0)

In [62]:
pivot_df

Month                                       2020-07  2020-08  2020-09  \
Part_Type   Part_Number Customer                                        
Air Filter  AF9039      Advance Auto Parts       13        0       11   
                        Amazon                    0        0        7   
                        AutoZone                  0       12        0   
                        CarQuest                 13        8       22   
                        NAPA                      0       11        0   
...                                             ...      ...      ...   
Wiper Blade WB3957      NAPA                      0        7        4   
                        O'Reilly                  0       18        5   
                        RockAuto                  2       24       11   
                        Summit Racing             3       30        7   
                        Walmart                   9       20       25   

Month                                       2020-10  2020-11  2020-12  \
Part_Type   Part_Number Customer                                        
Air Filter  AF9039      Advance Auto Parts        9       12       18   
                        Amazon                   23        5        0   
                        AutoZone                 25        4       13   
                        CarQuest                  4        1        6   
                        NAPA                     12        6        2   
...                                             ...      ...      ...   
Wiper Blade WB3957      NAPA                     22       29       19   
                        O'Reilly                 15       11        7   
                        RockAuto                  1       12        8   
                        Summit Racing             6       21       12   
                        Walmart                  14        8        8   

Month                                       2021-01  2021-02  2021-03  \
Part_Type   Part_Number Customer                                        
Air Filter  AF9039      Advance Auto Parts        4        0       19   
                        Amazon                   14       15        3   
                        AutoZone                 20        6       14   
                        CarQuest                  9       21        9   
                        NAPA                     14       24        7   
...                                             ...      ...      ...   
Wiper Blade WB3957      NAPA                      4       21        5   
                        O'Reilly                 21        0       16   
                        RockAuto                  1       27       12   
                        Summit Racing            14       22        6   
                        Walmart                  25       16        9   

Month                                       2021-04  ...  2024-10  2024-11  \
Part_Type   Part_Number Customer                     ...                     
Air Filter  AF9039      Advance Auto Parts        0  ...       10        0   
                        Amazon                    2  ...        0       12   
                        AutoZone                  0  ...       13        7   
                        CarQuest                 20  ...        1        0   
                        NAPA                     22  ...       18       15   
...                                             ...  ...      ...      ...   
Wiper Blade WB3957      NAPA                      9  ...       26        4   
                        O'Reilly                 17  ...       10        6   
                        RockAuto                  0  ...        0        0   
                        Summit Racing            21  ...       27        9   
                        Walmart                   9  ...       22        6   

Month                                       2024-12  2025-01  2025-02  \
Part_Type   Part_Number Customer                              

In [ ]:
# unique_parts = df_sales['Part_Number'].unique()
# cost_mapping = {part: round(random.uniform(10, 100), 2) for part in unique_parts}
# df_sales['Parts_Cost_USD'] = df_sales['Part_Number'].map(cost_mapping)

In [63]:
pivot_df.to_csv(rf"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\WarrantyData\Sales\Sales_data.csv", index=True)

In [64]:
df_sales_updated=pd.read_csv(rf"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\WarrantyData\Sales\Sales_data.csv")

In [65]:
PartTypes= df_sales['Part_Type'].unique().tolist()

In [66]:
for i in range (len(PartTypes)):
    df_sales_updated[df_sales_updated['Part_Type']==PartTypes[i]].to_csv(rf"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\WarrantyData\Sales\{PartTypes[i]}_SalesData.csv", index=False)
    print(f"✅ Dataset saved as '{PartTypes[i]}_WarrantyData.csv'")


✅ Dataset saved as 'Oil Filter_WarrantyData.csv'
✅ Dataset saved as 'Engine Water Pump_WarrantyData.csv'
✅ Dataset saved as 'Wiper Blade_WarrantyData.csv'
✅ Dataset saved as 'Ignition Coil_WarrantyData.csv'
✅ Dataset saved as 'Brake Rotor_WarrantyData.csv'
✅ Dataset saved as 'Air Filter_WarrantyData.csv'
✅ Dataset saved as 'Fuel Pump_WarrantyData.csv'
✅ Dataset saved as 'Spark Plug_WarrantyData.csv'
✅ Dataset saved as 'Tailgate Lift Support_WarrantyData.csv'
